

# ISMF Sharpe Quants Project
Team members: Seán Herlihy, Jack Kingston , Patrick Ryan

### Importing Libraries

In [ ]:
!pip install hurst # have to run each session , on cloud
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
import re # regex for filtering data
import hurst as hurst
from hurst import compute_Hc # Hurst exponent test

### Downloading Data

In [ ]:
start = '2020-01-01' # only using 1 year of data so code runs faster , can add more years for backtesting results
end = '2026-01-01'

basket =  ["ABT", "TMO", "DHR", "BDX"]
 # Enter tickers here


In [ ]:
raw_data = yf.download(basket, start, end)['Close']
raw_data = raw_data.ffill().dropna() # just downloading closing prices
# raw_data.head() # if you wanna take a look at data

In [ ]:
raw_data.columns = [f"A{i+1}" for i in range(len(raw_data.columns))] # renames all columns in the order you entered them
raw_data.columns

### Taking the log of prices and modeling them as a linear system. Then using regression to find optimal coefficients (Hedge Ratio)

In [ ]:
log_prices = np.log(raw_data)

Y = log_prices['A1']
X = log_prices.filter(regex = r"^A[2-9]\d*") # any column from 2 onwards
X = sm.add_constant(X)
X.head()

In [ ]:
linear_model = sm.OLS(Y,X).fit()
residuals = linear_model.resid # finds the residuals between model and actual
#residuals.tail()

### Tests for Mean reversion:
- Augmented Dickey Fuller: Tests for stationarity
- Half- Life test: Tests the speed of which mean reversion
- Hurst Exponent: Tests long term memory

In [ ]:

def adf_output(residuals):
  adf_test = adfuller(residuals)
  adf_stat, p_value  = adf_test[:2]
  if p_value < 0.01:
    strength = "Very Strong"
  elif p_value < 0.05:
    strength = "Strong"
  elif p_value < 0.1:
    strength = "Weak"
  else: strength = "FAIL"

  print(f"ADF Statistic: {adf_stat:.3f}")
  print(f"p-value:  {p_value:.3f}")
  print(f"Mean Reversion: {strength}")

def half_life_output(residuals):
  y_lag = sm.add_constant(residuals.shift(1).dropna())
  delta_y = residuals.diff().dropna()
  mod = sm.OLS(delta_y, y_lag).fit()
  half_life = -np.log(2) / mod.params.iloc[1]
  print(f"Half Life: {half_life}")


def hurst_exponent_output(residuals):
    ts = np.array(residuals.dropna(), dtype=float)
    H, c, data = compute_Hc(ts, kind='change', simplified=True) # built in library way easier

    strength = "Residuals are Mean Reverting " if H < 0.5 else ("Random Walk" if H == 0.5 else "Trending, Not mean reverting")
    print(f"Hurst Exponent: {H:.3f}")
    print(f"Interpretation: {strength}")

In [ ]:
def test_statistics(residuals):
  print("Mean Reversion Summary")
  print("-" * 30)
  adf_output(residuals)
  print("-" * 30)
  half_life_output(residuals)
  print("-" * 30)
  # Added check for residuals length before calling hurst_exponent_output
  if len(residuals.dropna()) >= 100:
    hurst_exponent_output(residuals)
  else:
    print(f"Hurst Exponent: Cannot be calculated. Series length ({len(residuals.dropna())}) must be >= 100.")

test_statistics(residuals)

### Trading Signals
- We use a rolling window to calculate a mean and standard deviation to standardise our residuals

- When z is below -entry_z : Long A1, Short βᵢAᵢ
- When z is above entry z: Short A1, Long βᵢAᵢ
- Exit position when the spread has mean-reverted (near 0)

In [ ]:
window = 30

# Uses all available data up to day t, then switches to rolling once window is full
res_mean = residuals.expanding(min_periods=2).mean()
res_std  = residuals.expanding(min_periods=2).std()

# Once you have 30 days of data, switch to rolling
roll_mean = residuals.rolling(window).mean()
roll_std  = residuals.rolling(window).std()

res_mean = roll_mean.fillna(res_mean)   # fill NaNs from rolling with expanding values
res_std  = roll_std.fillna(res_std)

z_score = (residuals - res_mean) / res_std

entry_point = 1.5
exit_point = 0.5

trades = pd.Series(0 ,index = residuals.index)
trades[z_score < -entry_point] = 1
trades[z_score > entry_point] = -1
positions = trades.copy().fillna(method = 'ffill')

positions[np.abs(z_score) < exit_point] = 0

positions = positions.fillna(method='ffill').fillna(0)
# pd.concat([residuals[10:30], z_score[10:30] , positions[10:30]], axis=1).rename(columns={0:'Spread',1:'Z',2:'Pos'}) #taking a look at data , looks good


### Profit and Loss
- If position = 1:  return = Return_A1*A1  - Βᵢ x Return_Aᵢ

In [ ]:
returns = raw_data.pct_change()

betas = linear_model.params.drop('const') # extracting hedge ratios for PnL calculation

# Hedge weights: long A1, short A2,A3,... by their beta amounts
weights = np.concatenate([[1.0], -betas.values])
print(f"Weights: {weights}")

# print(linear_model.summary()) # Can Check betas manually

In [ ]:
spread_returns   = returns @ weights                        # single spread return series
strategy_returns = spread_returns * positions.shift(1)     # apply positions with lookahead guard
strategy_returns = strategy_returns.fillna(0)


In [ ]:
import matplotlib.dates as mdates

cumulative_strategy_returns = (1 + strategy_returns).cumprod()

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))

# Plot 1
ax1.plot((cumulative_strategy_returns - 1) * 100, color='darkblue')
ax1.axhline(y=1, color='grey', linewidth=0.5, linestyle='--')
ax1.grid(axis='x', color='grey', linewidth=0.4, linestyle='--', alpha=0.4)
ax1.set_ylabel("Cumulative Returns")
ax1.set_title("Cumulative Strategy Returns")

ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"+{x:.0f}%" if x > 0 else f"{x:.0f}%"))

# Plot 2
ax2.plot(residuals, color = 'lightgreen')
ax2.axhline(y=0, color='grey', linewidth=0.5, linestyle='--')
ax2.set_title("Log(Spread) over time")

# ax3

ax3.plot(z_score,color = 'orange')
ax3.axhline(y=0, color='grey', linewidth=0.5, linestyle='--')
ax3.set_title("Residual Z-scores Over time ")
ax3.axhline(y= entry_point, color='green', linewidth=0.5, linestyle='--',label = 'Entry')
ax3.axhline(y= -entry_point, color='green', linewidth=0.5, linestyle='--')
ax3.axhline(y= exit_point, color='red', linewidth=0.5, linestyle='--',label = 'Exit')
ax3.axhline(y= -exit_point, color='red', linewidth=0.5, linestyle='--')
plt.legend()

# --- Shared formatting ---
for ax in (ax1, ax2, ax3):
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.tick_params(axis='x', labelsize=7, rotation=45)


plt.tight_layout()

#### Diagnostics of returns
- Sharpe Ratio
- Max Drawdown
- Final Return (no costs)
- Total Trades made

In [ ]:
#Diagnostic functions
def sharpe_ratio(returns, rf=0.0): # rf is risk free rate
    # returns: daily returns series
    excess = returns - rf/252 # Diff between our return and risk free daily
    sharpe_ratio = np.sqrt(252) * (excess.mean() / (excess.std() + 1e-20)) # add a small number so dont divide by 0 and annualize
    if sharpe_ratio < 1: strength = "Weak"
    elif sharpe_ratio < 2: strength = "Good"
    elif sharpe_ratio < 3: strength = "Strong"
    else: strength = "Incredible , TAKE CAUTION (overfitting)"
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"Interpretation: {strength}")


def max_drawdown(cum_returns):
    roll_max = cum_returns.cummax()
    drawdown = cum_returns / roll_max - 1.0
    return drawdown.min()


md = max_drawdown(cumulative_strategy_returns)
total_return = cumulative_strategy_returns.iloc[-1] - 1 # final entry - 1
trade_changes = (positions.diff().fillna(0) != 0).sum()

sharpe_ratio(strategy_returns)
print(f"Max Drawdown: {md:.2%}")
print(f"Final Return (strategy): {total_return:.2%}")
print(f"Approx number of position changes: {int(trade_changes)}")